## Data Setup

Loading required data sources for visualizations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, sum, avg, round, when, countDistinct, stddev, max, min, date_format, lower
from pyspark.sql.window import Window

# Load base tables from Silver layer (needed for incident-level detail)
df_operations = spark.table("vattenfall_dev.refined.silver_regional_operations_base")
df_market_prices = spark.table("vattenfall_dev.refined.silver_market_prices")
df_weather = spark.table("vattenfall_dev.refined.silver_weather")

print(f"✅ Loaded {df_operations.count()} operational records from Silver")
print(f"✅ Loaded {df_market_prices.count()} market price observations from Silver")
print(f"✅ Loaded {df_weather.count()} weather observations from Silver")

In [0]:
# Create incidents_weather dataframe by joining operations with weather

# Step 1: Add city → country mapping to events
events_by_date = df_operations \
    .withColumn("event_date", F.to_date("timestamp")) \
    .withColumn(
        "country_code",
        F.when(F.col("region_name").isin("Helsinki", "Turku", "Tampere"), "FI")
         .when(F.col("region_name") == "Copenhagen", "DK")
         .when(F.col("region_name").isin("Gothenburg", "Malmo"), "SE")
         .otherwise(None)
    )

# Step 2: Aggregate weather to DAILY level
weather_daily = df_weather \
    .groupBy("report_day", F.lower(col("region_normalized")).alias("weather_country")).agg(
        F.round(F.avg("temperature_c"), 1).alias("avg_temp_c"),
        F.round(F.avg("wind_speed_ms"), 1).alias("avg_wind_speed_ms"),
        F.round(F.avg("precipitation_mm"), 2).alias("avg_precipitation_mm"),
        F.round(F.avg("cloud_cover_percent"), 1).alias("avg_cloud_cover_pct")
    ) \
    .withColumn(
        "is_cold",
        when(col("avg_temp_c") < -5, 1).otherwise(0)
    ) \
    .withColumn(
        "is_high_wind",
        when(col("avg_wind_speed_ms") > 15, 1).otherwise(0)
    ) \
    .withColumn(
        "has_precipitation",
        when(col("avg_precipitation_mm") > 0, 1).otherwise(0)
    )

# Step 3: Join operations with weather data
incidents_weather = df_operations \
    .withColumn("event_date", F.to_date("timestamp")) \
    .withColumn(
        "country_code",
        F.when(F.col("region_name").isin("Helsinki", "Turku", "Tampere"), "FI")
         .when(F.col("region_name") == "Copenhagen", "DK")
         .when(F.col("region_name").isin("Gothenburg", "Malmo"), "SE")
         .otherwise(None)
    ) \
    .join(
        weather_daily,
        (F.to_date(df_operations.timestamp) == weather_daily.report_day) &
        (F.lower(F.col("country_code")) == weather_daily.weather_country),
        "left"
    )

print("✅ Created incidents_weather dataset with weather data joined")
print(f"   Records: {incidents_weather.count()}")

---

# 🎯 Key Grid Insights: Answering Critical Questions

## Questions We'll Answer:

1. **Why does Copenhagen have the highest risk score?** (And what is it composed of?)
2. **Why are SUB128, SUB149, and SUB112 the worst performers?** (And why is SUB121 bad despite being only 10 years old?)
3. **How do weather conditions affect grid events?**
4. **Are weather conditions the only direct cause?**
5. **What are the operational ratings for each country?**

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

# Professional styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

print("✅ Visualization setup complete")

## 🔴 Question 1: Why Does Copenhagen Have the Highest Risk Score?

**Risk Score Formula:**
```
Risk Score = (Aging Assets × 2) + (High-Risk Events × 3) + (Avg Asset Age × 0.5) + (Avg Duration × 0.1)
```

**What we'll discover:** Which component contributes most to Copenhagen's risk?

In [0]:
# Calculate risk score components for all regions
from pyspark.sql.functions import col, when, sum as _sum, avg as _avg, round as _round, countDistinct

risk_components = df_operations.groupBy("region_name").agg(
    countDistinct("substation_id").alias("total_substations"),
    _sum(when(col("asset_age_category") == "aging", 1).otherwise(0)).alias("aging_substations"),
    _sum(when(col("high_risk_event") == True, 1).otherwise(0)).alias("high_risk_events"),
    _round(_avg("asset_age_years"), 2).alias("avg_asset_age_years"),
    _round(_avg("duration_minutes"), 1).alias("avg_duration_min")
).toPandas()

# Calculate each component's contribution
risk_components['aging_score'] = risk_components['aging_substations'] * 2
risk_components['risk_events_score'] = risk_components['high_risk_events'] * 3
risk_components['age_score'] = risk_components['avg_asset_age_years'] * 0.5
risk_components['duration_score'] = risk_components['avg_duration_min'] * 0.1
risk_components['total_risk_score'] = (
    risk_components['aging_score'] + 
    risk_components['risk_events_score'] + 
    risk_components['age_score'] + 
    risk_components['duration_score']
).round(1)

# Sort by total risk score descending
risk_components = risk_components.sort_values('total_risk_score', ascending=False)

# Create stacked bar chart with CALMER COLORS
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(risk_components))
width = 0.6

# Stack the components with calmer colors
p1 = ax.bar(x, risk_components['aging_score'], width, label='Aging Assets (×2)', color='#B5876E')
p2 = ax.bar(x, risk_components['risk_events_score'], width, bottom=risk_components['aging_score'],
            label='High-Risk Events (×3)', color='#C19A7F')
p3 = ax.bar(x, risk_components['age_score'], width, 
            bottom=risk_components['aging_score'] + risk_components['risk_events_score'],
            label='Average Age (×0.5)', color='#C9B88A')
p4 = ax.bar(x, risk_components['duration_score'], width,
            bottom=risk_components['aging_score'] + risk_components['risk_events_score'] + risk_components['age_score'],
            label='Restoration Duration (×0.1)', color='#7BA7BC')

# Add total score labels on top
for i, (idx, row) in enumerate(risk_components.iterrows()):
    total = row['total_risk_score']
    ax.text(i, total + 1.5, f"{total}", ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylabel('Risk Score Points', fontsize=13, fontweight='bold')
ax.set_xlabel('Region', fontsize=13, fontweight='bold')
ax.set_title('Risk Score Composition by Region: Copenhagen\'s Duration Problem', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(risk_components['region_name'], fontsize=12)
ax.legend(loc='upper right', framealpha=0.95, fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed breakdown
print("\n" + "="*80)
print("RISK SCORE BREAKDOWN BY REGION")
print("="*80)
for idx, row in risk_components.iterrows():
    print(f"\n{row['region_name']}:")
    print(f"  Total Risk Score: {row['total_risk_score']}")
    print(f"  - Aging Assets: {row['aging_substations']} substations × 2 = {row['aging_score']} pts ({row['aging_score']/row['total_risk_score']*100:.1f}%)")
    print(f"  - High-Risk Events: {row['high_risk_events']} events × 3 = {row['risk_events_score']} pts ({row['risk_events_score']/row['total_risk_score']*100:.1f}%)")
    print(f"  - Avg Asset Age: {row['avg_asset_age_years']} years × 0.5 = {row['age_score']:.1f} pts ({row['age_score']/row['total_risk_score']*100:.1f}%)")
    print(f"  - Avg Duration: {row['avg_duration_min']} min × 0.1 = {row['duration_score']:.1f} pts ({row['duration_score']/row['total_risk_score']*100:.1f}%)")

print("\n" + "="*80)
print("🎯 KEY FINDING:")
cph = risk_components[risk_components['region_name'] == 'Copenhagen'].iloc[0]
print(f"Copenhagen's restoration duration ({cph['avg_duration_min']} min) contributes")
print(f"{cph['duration_score']:.1f} points = {cph['duration_score']/cph['total_risk_score']*100:.1f}% of its total risk score!")
print("="*80)

## ⚠️ Question 2: Why Are SUB128, SUB149, and SUB112 the Worst Performers?

**Performance Score Factors:**
- Incident frequency
- Restoration time
- Customer impact
- Asset age
- Critical incidents

**Special Investigation:** Why is SUB121 a bad performer despite being only 10 years old?

In [0]:
# Analyze all substations
substation_analysis = df_operations.groupBy(
    "substation_id", "region_name", "asset_age_years", "asset_age_category"
).agg(
    F.count("*").alias("incident_count"),
    F.sum("affected_customers").alias("total_customers"),
    F.round(F.avg("duration_minutes"), 1).alias("avg_duration_min"),
    F.sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_incidents"),
    F.max(when(col("high_risk_event") == True, 1).otherwise(0)).alias("has_high_risk")
).withColumn(
    "performance_score",
    F.round(
        (col("incident_count") * 10) + 
        (col("avg_duration_min") * 0.5) + 
        (col("total_customers") / 100) + 
        (col("critical_incidents") * 15) +
        (col("has_high_risk") * 20) +
        (col("asset_age_years") * 0.5),
        1
    )
).orderBy(col("performance_score").desc()).toPandas()

# Get top 10 worst performers
top10_worst = substation_analysis.head(10).copy()

# Identify the key substations
worst_3 = ['SUB128', 'SUB149', 'SUB112']
sub121 = 'SUB121'

# Define CALMER colors for regions
region_colors = {
    'Copenhagen': '#C19A7F',
    'Helsinki': '#B5876E', 
    'Turku': '#C9B88A',
    'Gothenburg': '#7BA7BC',
    'Tampere': '#9B8FB8',
    'Malmo': '#8BB896'
}

# Create single visualization - Age vs Performance Scatter
fig, ax = plt.subplots(figsize=(14, 10))

# Scatter plot with age on x-axis, score on y-axis for top 10
for _, row in top10_worst.iterrows():
    sub_id = row['substation_id']
    
    # Special highlighting
    if sub_id == sub121:
        marker = 'D'
        size = 500
        edgecolor = 'black'
        linewidth = 3
        alpha = 1.0
        zorder = 10
    elif sub_id in worst_3:
        marker = 'X'
        size = 400
        edgecolor = 'black'
        linewidth = 2.5
        alpha = 1.0
        zorder = 9
    else:
        marker = 'o'
        size = 250
        edgecolor = 'gray'
        linewidth = 1.5
        alpha = 0.7
        zorder = 5
    
    ax.scatter(row['asset_age_years'], row['performance_score'], 
               s=size, color=region_colors.get(row['region_name'], '#95A5A6'),
               marker=marker, alpha=alpha, edgecolors=edgecolor, linewidth=linewidth, zorder=zorder)
    
    # Label each substation
    ax.text(row['asset_age_years'], row['performance_score'] + 8, 
            f"{sub_id}\n{int(row['asset_age_years'])} yrs",
            fontsize=10, fontweight='bold', ha='center')

ax.set_xlabel('Asset Age (years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Performance Score (higher = worse)', fontsize=13, fontweight='bold')
ax.set_title('Age vs Performance: The SUB121 Anomaly', fontsize=16, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)

# Add vertical line at 20 years (aging threshold)
ax.axvline(x=20, color='#C19A7F', linestyle='--', linewidth=2, alpha=0.5, label='Typical aging threshold')
ax.legend(loc='upper left', fontsize=11)

# Highlight SUB121 paradox with annotation
sub121_row = top10_worst[top10_worst['substation_id'] == sub121]
if not sub121_row.empty:
    row = sub121_row.iloc[0]
    ax.annotate('SUB121: Only 10 years old\nbut 2nd worst performer!\n(Operational issue)', 
                xy=(row['asset_age_years'], row['performance_score']),
                xytext=(row['asset_age_years'] - 8, row['performance_score'] + 50),
                fontsize=11, fontweight='bold', color='#5D8AA8',
                bbox=dict(boxstyle='round', facecolor='#E8EEF2', alpha=0.95, edgecolor='#7BA7BC', linewidth=2),
                arrowprops=dict(arrowstyle='->', lw=2, color='#7BA7BC'))

# Add legend for markers
ax.text(0.02, 0.98, '⚠️ = User-specified worst (SUB128, SUB149, SUB112)\n💎 = SUB121 (Modern but poor performance)', 
         transform=ax.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='#F5F5F5', alpha=0.9, edgecolor='gray', linewidth=1))

plt.tight_layout()
plt.show()

# Print analysis of key substations
print("\n" + "="*80)
print("TOP 10 WORST PERFORMING SUBSTATIONS")
print("="*80)
print(f"\n{'Rank':<6} {'Substation':<12} {'Region':<12} {'Score':<8} {'Age':<6} {'Rest.Time':<12} {'Customers':<10}")
print("-" * 80)

for idx, (_, row) in enumerate(top10_worst.iterrows(), 1):
    marker = "⚠️" if row['substation_id'] in worst_3 else "💎" if row['substation_id'] == sub121 else "  "
    print(f"{marker} #{idx:<4} {row['substation_id']:<12} {row['region_name']:<12} "
          f"{row['performance_score']:<8.1f} {int(row['asset_age_years']):<6} "
          f"{row['avg_duration_min']:<12.0f} {int(row['total_customers']):<10,}")

print("\n" + "="*80)
print("🎯 KEY FINDINGS")
print("="*80)

print("\n1. SUB128 (Copenhagen) - THE WORST:")
sub128 = top10_worst[top10_worst['substation_id'] == 'SUB128']
if not sub128.empty:
    s = sub128.iloc[0]
    print(f"   - Score: {s['performance_score']:.1f} (Rank #1)")
    print(f"   - Age: {int(s['asset_age_years'])} years (aging asset)")
    print(f"   - Restoration: {s['avg_duration_min']:.0f} minutes (catastrophic)")
    print(f"   - Customers: {int(s['total_customers']):,}")
    print(f"   → Both aging infrastructure AND operational problems")

print("\n2. SUB121 (Gothenburg) - THE PARADOX:")
sub121_data = top10_worst[top10_worst['substation_id'] == sub121]
if not sub121_data.empty:
    s = sub121_data.iloc[0]
    rank = top10_worst[top10_worst['substation_id'] == sub121].index[0] + 1
    print(f"   - Score: {s['performance_score']:.1f} (Rank #{rank})")
    print(f"   - Age: {int(s['asset_age_years'])} years (MODERN ASSET!)")
    print(f"   - Restoration: {s['avg_duration_min']:.0f} minutes")
    print(f"   - Customers: {int(s['total_customers']):,}")
    print(f"   → This is 100% OPERATIONAL FAILURE, not asset age!")
    print(f"   → Despite being modern, performs WORSE than older assets")

print("\n3. SUB149 & SUB112 (Helsinki):")
for sub_id in ['SUB149', 'SUB112']:
    sub_data = top10_worst[top10_worst['substation_id'] == sub_id]
    if not sub_data.empty:
        s = sub_data.iloc[0]
        rank = top10_worst[top10_worst['substation_id'] == sub_id].index[0] + 1
        print(f"   - {sub_id}: Score {s['performance_score']:.1f} (Rank #{rank}), "
              f"{int(s['asset_age_years'])} yrs, {s['avg_duration_min']:.0f} min restoration")

print("\n4. THE BIG LESSON:")
print("   Age is NOT the primary predictor of poor performance!")
print("   Modern equipment with poor operations > Old equipment with good operations")
print("   → Fix operational issues (training, protocols) BEFORE replacing assets")
print("="*80)

## 🔍 Question 3: Are Weather Conditions the Only Direct Cause?

**Investigation:**
- Weather-only incidents (modern assets)
- Weather + Aging asset incidents  
- Other factors

**What we'll discover:** Is weather sufficient, or do other factors amplify the impact?

In [0]:
# Categorize incidents by causation
incidents_categorized = incidents_weather.withColumn(
    "has_adverse_weather",
    when((col("is_cold") == 1) | (col("is_high_wind") == 1) | (col("has_precipitation") == 1), 1).otherwise(0)
).withColumn(
    "is_aging_asset",
    when(col("asset_age_category") == "aging", 1).otherwise(0)
).withColumn(
    "cause_category",
    when((col("has_adverse_weather") == 1) & (col("is_aging_asset") == 1), "Weather + Aging Asset")
    .when(col("has_adverse_weather") == 1, "Weather Only")
    .when(col("is_aging_asset") == 1, "Aging Asset Only")
    .otherwise("Other Factors")
)

# Aggregate by cause
cause_summary = incidents_categorized.groupBy("cause_category").agg(
    F.count("*").alias("incident_count"),
    _round(_avg("duration_minutes"), 1).alias("avg_restoration_min"),
    _round(_avg("affected_customers"), 0).alias("avg_customers"),
    _sum("affected_customers").alias("total_customers")
).orderBy(col("incident_count").desc()).toPandas()

# Create single bar chart visualization with CALMER COLORS
fig, ax = plt.subplots(figsize=(14, 8))

x_pos = np.arange(len(cause_summary))
colors_bar = ['#C9B88A', '#9B8FB8', '#8BB896', '#7BA7BC'][:len(cause_summary)]

bars = ax.bar(x_pos, cause_summary['avg_restoration_min'], color=colors_bar, edgecolor='white', linewidth=2)
ax.set_ylabel('Average Restoration Time (minutes)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cause Category', fontsize=13, fontweight='bold')
ax.set_title('Incident Severity by Cause: Which Failures Take Longer?', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x_pos)
ax.set_xticklabels(cause_summary['cause_category'], rotation=15, ha='right', fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars with incident count
for i, (bar, val, count) in enumerate(zip(bars, cause_summary['avg_restoration_min'], cause_summary['incident_count'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 5,
             f'{val:.0f} min\n({count} incidents)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("INCIDENT CAUSATION BREAKDOWN")
print("="*80)

for idx, row in cause_summary.iterrows():
    pct = (row['incident_count'] / cause_summary['incident_count'].sum()) * 100
    print(f"\n{row['cause_category']}:")
    print(f"  - Incidents: {row['incident_count']} ({pct:.0f}%)")
    print(f"  - Avg restoration: {row['avg_restoration_min']:.0f} min")
    print(f"  - Avg customers: {int(row['avg_customers']):,}")
    print(f"  - Total customers: {int(row['total_customers']):,}")

print("\n" + "="*80)
print("🎯 KEY INSIGHTS:")
print("="*80)

weather_incidents = cause_summary[cause_summary['cause_category'].str.contains('Weather')]['incident_count'].sum()
total = cause_summary['incident_count'].sum()
weather_pct = (weather_incidents / total) * 100
print(f"\n• {weather_pct:.0f}% of incidents involve adverse weather")

aging_incidents = cause_summary[cause_summary['cause_category'].str.contains('Aging')]['incident_count'].sum()
aging_pct = (aging_incidents / total) * 100
print(f"• {aging_pct:.0f}% involve aging infrastructure")

combined = cause_summary[cause_summary['cause_category'] == 'Weather + Aging Asset']['incident_count'].sum() if 'Weather + Aging Asset' in cause_summary['cause_category'].values else 0
if combined > 0:
    print(f"• {combined} incidents show COMPOUNDING RISK (weather + aging)")

print("="*80)

## 🏏 Question 5: What Are the Operational Ratings for Each Country?

**Rating Scale:**
- ⭐ **EXCELLENT** (80-100): Optimal conditions
- 🟢 **GOOD** (60-80): Low incident activity  
- 🟡 **FAIR** (40-60): Moderate stress
- 🟠 **POOR** (20-40): Elevated incidents
- 🔴 **CRITICAL** (0-20): Major incidents

**What we'll discover:** How does each country perform overall?

In [0]:
# Load daily ratings from gold layer
daily_ratings = spark.table("vattenfall_dev.gold.gold_regional_condition_daily").select(
    "report_day", "region", "operational_condition", "health_score", 
    "total_incident_count", "total_customers_affected"
).toPandas()

# Map region codes to country names
region_to_country = {
    'DK': 'Denmark',
    'FI': 'Finland',
    'SE': 'Sweden',
    'NO': 'Norway'
}

daily_ratings['country'] = daily_ratings['region'].map(region_to_country)

# Calculate country-level metrics
country_summary = daily_ratings.groupby('country').agg({
    'health_score': 'mean',
    'total_incident_count': 'sum',
    'total_customers_affected': 'sum',
    'report_day': 'count'
}).reset_index()

country_summary.columns = ['country', 'avg_health_score', 'total_incidents', 'total_customers', 'num_region_days']
country_summary = country_summary.sort_values('avg_health_score', ascending=False)

# Create single bar chart visualization with distinct colors per country
fig, ax = plt.subplots(figsize=(14, 8))

# Assign distinct colors from the calmer palette
country_colors = {
    'Denmark': '#7BA7BC',
    'Finland': '#B5876E',
    'Sweden': '#C9B88A',
    'Norway': '#9B8FB8'
}

colors = [country_colors.get(country, '#95A5A6') for country in country_summary['country']]

bars = ax.bar(country_summary['country'], country_summary['avg_health_score'], 
              color=colors, edgecolor='white', linewidth=2)
ax.axhline(y=60, color='#8BB896', linestyle='--', linewidth=2, alpha=0.5, label='GOOD threshold (60)')
ax.axhline(y=40, color='#C9B88A', linestyle='--', linewidth=2, alpha=0.5, label='FAIR threshold (40)')
ax.axhline(y=20, color='#C19A7F', linestyle='--', linewidth=2, alpha=0.5, label='CRITICAL threshold (20)')
ax.set_ylabel('Average Health Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Country', fontsize=13, fontweight='bold')
ax.set_title('Country Operational Health Scores', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 80)
ax.legend(loc='upper right', fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels with rating
for bar, score in zip(bars, country_summary['avg_health_score']):
    height = bar.get_height()
    rating = 'GOOD' if score >= 60 else 'FAIR' if score >= 40 else 'POOR' if score >= 20 else 'CRITICAL'
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{score:.1f}\n({rating})',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("COUNTRY OPERATIONAL HEALTH SUMMARY")
print("="*80)

for _, row in country_summary.iterrows():
    rating = 'EXCELLENT' if row['avg_health_score'] >= 80 else 'GOOD' if row['avg_health_score'] >= 60 else 'FAIR' if row['avg_health_score'] >= 40 else 'POOR' if row['avg_health_score'] >= 20 else 'CRITICAL'
    emoji = '⭐' if rating == 'EXCELLENT' else '🟢' if rating == 'GOOD' else '🟡' if rating == 'FAIR' else '🟠' if rating == 'POOR' else '🔴'
    
    print(f"\n{emoji} {row['country']}:")
    print(f"   Rating: {rating}")
    print(f"   Health Score: {row['avg_health_score']:.1f}")
    print(f"   Total Incidents: {int(row['total_incidents'])}")
    print(f"   Customers Affected: {int(row['total_customers']):,}")
    print(f"   Days Tracked: {int(row['num_region_days'])}")

print("\n" + "="*80)
print("🎯 KEY INSIGHTS:")
print("="*80)
best = country_summary.iloc[0]
worst = country_summary.iloc[-1]
print(f"\n• Best Performer: {best['country']} (Health Score: {best['avg_health_score']:.1f})")
print(f"• Needs Attention: {worst['country']} (Health Score: {worst['avg_health_score']:.1f})")
print(f"• Average across all countries: {country_summary['avg_health_score'].mean():.1f}")
print("="*80)

## ⏰ Question 6: When Do Grid Failures Happen?

**Temporal Analysis:**
- Hour of day patterns (peak demand correlation?)
- Weekend vs weekday response
- Trends over time (improving or degrading?)
- Day clustering (cascade effects?)

**What we'll discover:** Are failures predictable based on time patterns?

In [0]:
# Analyze temporal patterns
temporal_analysis = df_operations.select(
    "event_id", "timestamp", "region_name", "duration_minutes", 
    "affected_customers", "severity", "hour", "day_of_week", "is_weekend", "event_day"
).toPandas()

temporal_analysis['timestamp'] = pd.to_datetime(temporal_analysis['timestamp'])
temporal_analysis['event_day'] = pd.to_datetime(temporal_analysis['event_day'])

# Create single focused visualization - Hour of Day Distribution with CALMER COLORS
fig, ax = plt.subplots(figsize=(14, 8))

hour_counts = temporal_analysis.groupby('hour').size()
hours = range(24)
counts = [hour_counts.get(h, 0) for h in hours]

# Color code by time of day with calmer colors
colors_hour = ['#9B8FB8' if h < 6 or h >= 22 else '#8BB896' if h < 12 else '#C9B88A' if h < 18 else '#B5876E' 
               for h in hours]

ax.bar(hours, counts, color=colors_hour, edgecolor='white', linewidth=2)
ax.set_xlabel('Hour of Day (24h format)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Incidents', fontsize=13, fontweight='bold')
ax.set_title('Incident Distribution by Hour of Day', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(range(0, 24, 2))
ax.grid(axis='y', alpha=0.3)

# Add time period labels
ax.text(0.02, 0.98, 'Night (22-6): Purple\nMorning (6-12): Green\nAfternoon (12-18): Tan\nEvening (18-22): Brown', 
         transform=ax.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='#F5F5F5', alpha=0.9, edgecolor='gray', linewidth=1))

# Highlight peak hours
if len(hour_counts) > 0:
    peak_hour = hour_counts.idxmax()
    peak_count = hour_counts[peak_hour]
    ax.text(peak_hour, peak_count + 0.3, f'Peak\n{peak_hour}:00', 
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#C19A7F')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("TEMPORAL ANALYSIS")
print("="*80)

print("\n1. HOUR OF DAY PATTERNS:")
if len(hour_counts) > 0:
    peak_hour = hour_counts.idxmax()
    print(f"   Peak incident hour: {peak_hour}:00 ({hour_counts[peak_hour]} incidents)")
    
night_incidents = int(np.sum([hour_counts.get(h, 0) for h in list(range(22, 24)) + list(range(0, 6))]))
day_incidents = int(np.sum([hour_counts.get(h, 0) for h in range(6, 22)]))
print(f"   Night incidents (22:00-6:00): {night_incidents} ({night_incidents/len(temporal_analysis)*100:.0f}%)")
print(f"   Day incidents (6:00-22:00): {day_incidents} ({day_incidents/len(temporal_analysis)*100:.0f}%)")

print("\n2. WEEKEND VS WEEKDAY:")
weekday_data = temporal_analysis[temporal_analysis['is_weekend'] == 0]
weekend_data = temporal_analysis[temporal_analysis['is_weekend'] == 1]
print(f"   Weekday incidents: {len(weekday_data)} ({len(weekday_data)/len(temporal_analysis)*100:.0f}%)")
print(f"   Weekend incidents: {len(weekend_data)} ({len(weekend_data)/len(temporal_analysis)*100:.0f}%)")
if len(weekday_data) > 0 and len(weekend_data) > 0:
    weekday_avg_duration = weekday_data['duration_minutes'].mean()
    weekend_avg_duration = weekend_data['duration_minutes'].mean()
    print(f"   Weekday avg duration: {weekday_avg_duration:.0f} min")
    print(f"   Weekend avg duration: {weekend_avg_duration:.0f} min")
    if weekend_avg_duration > weekday_avg_duration:
        print(f"   → Weekend incidents take {weekend_avg_duration/weekday_avg_duration:.1f}x longer to resolve!")

print("\n3. KEY INSIGHT:")
if day_incidents > night_incidents:
    print(f"   Most incidents occur during daytime hours, suggesting operational load correlation")
else:
    print(f"   Significant night-time incidents suggest infrastructure stress independent of demand")
print("="*80)

## 💰 Question 10: Market Price vs Grid Stress

**Economic Impact Analysis:**
- Do failures happen when electricity is most expensive?
- Lost revenue calculation
- Price volatility during incidents
- Economic opportunity cost

**What we'll discover:** The financial pain of poor timing

In [0]:
# Load and prepare market price data
market_prices_pd = df_market_prices.toPandas()
market_prices_pd['report_day'] = pd.to_datetime(market_prices_pd['report_day'])
market_prices_pd['timestamp'] = pd.to_datetime(market_prices_pd['timestamp'])

# Map region codes to country names
region_to_country = {
    'DK': 'Denmark',
    'FI': 'Finland',
    'SE': 'Sweden',
    'NO': 'Norway'
}
market_prices_pd['country'] = market_prices_pd['region_normalized'].map(region_to_country)

# Prepare operations data
operations_pd = df_operations.toPandas()
operations_pd['timestamp'] = pd.to_datetime(operations_pd['timestamp'])
operations_pd['event_date'] = pd.to_datetime(operations_pd['timestamp']).dt.date

# Map regions to countries for price matching
region_to_country_ops = {
    'Copenhagen': 'Denmark',
    'Helsinki': 'Finland',
    'Turku': 'Finland',
    'Gothenburg': 'Sweden',
    'Malmo': 'Sweden',
    'Tampere': 'Finland'
}
operations_pd['country'] = operations_pd['region_name'].map(region_to_country_ops)

# Calculate daily average prices by country
daily_avg_prices = market_prices_pd.groupby(
    [market_prices_pd['report_day'].dt.date, 'country']
)['price_eur_mwh'].mean().reset_index()
daily_avg_prices.columns = ['date', 'country', 'avg_price']

# Count daily incidents by country
daily_incidents = operations_pd.groupby(['event_date', 'country']).size().reset_index(name='incident_count')

# Merge prices with incidents
price_incident_analysis = daily_avg_prices.merge(
    daily_incidents,
    left_on=['date', 'country'],
    right_on=['event_date', 'country'],
    how='outer'
).fillna({'incident_count': 0, 'avg_price': np.nan})

price_incident_analysis['date'] = pd.to_datetime(price_incident_analysis['date'])
price_incident_analysis = price_incident_analysis.sort_values('date')

# Calculate lost revenue
incidents_with_prices = operations_pd.merge(
    daily_avg_prices,
    left_on=['event_date', 'country'],
    right_on=['date', 'country'],
    how='left'
)

# Assumption: Average household consumption is 1 kWh/hour
incidents_with_prices['energy_not_delivered_mwh'] = (
    incidents_with_prices['affected_customers'] * incidents_with_prices['duration_minutes']
) / (1000 * 60)

incidents_with_prices['lost_revenue_eur'] = (
    incidents_with_prices['energy_not_delivered_mwh'] * incidents_with_prices['avg_price']
)

# Create single focused dual-axis chart with CALMER COLORS
fig, ax1 = plt.subplots(figsize=(16, 8))

# Aggregate by date across all countries
daily_summary = price_incident_analysis.groupby('date').agg({
    'avg_price': 'mean',
    'incident_count': 'sum'
}).reset_index()

# Primary axis: Price trend (area chart)
color_price = '#C9B88A'
ax1.fill_between(daily_summary['date'], daily_summary['avg_price'], 
                 alpha=0.5, color=color_price, edgecolor=color_price, linewidth=2)
ax1.plot(daily_summary['date'], daily_summary['avg_price'], 
         color=color_price, linewidth=2.5, label='Avg Price (EUR/MWh)')
ax1.set_xlabel('Date (January 2024)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Average Electricity Price (EUR/MWh)', fontsize=13, fontweight='bold', color=color_price)
ax1.tick_params(axis='y', labelcolor=color_price)
ax1.grid(alpha=0.2)

# Secondary axis: Incident count (bar chart)
ax2 = ax1.twinx()
color_incidents = '#B5876E'
ax2.bar(daily_summary['date'], daily_summary['incident_count'], 
        alpha=0.6, color=color_incidents, edgecolor='white', linewidth=1.5, 
        width=0.8, label='Incident Count')
ax2.set_ylabel('Number of Incidents', fontsize=13, fontweight='bold', color=color_incidents)
ax2.tick_params(axis='y', labelcolor=color_incidents)

# Title
ax1.set_title('Market Price vs Grid Stress: Do Failures Happen at Expensive Times?', 
             fontsize=16, fontweight='bold', pad=20)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("MARKET PRICE vs GRID STRESS ANALYSIS")
print("="*80)

print("\n1. TIMING CORRELATION:")
# Calculate correlation between price and incident frequency
if len(daily_summary[daily_summary['incident_count'] > 0]) > 1:
    corr_data = daily_summary.dropna(subset=['avg_price', 'incident_count'])
    if len(corr_data) > 1:
        correlation = corr_data['avg_price'].corr(corr_data['incident_count'])
        print(f"   Price-Incident correlation: {correlation:.3f}")
        if abs(correlation) < 0.3:
            print(f"   → WEAK correlation: Failures appear independent of electricity price")
        elif correlation > 0:
            print(f"   → POSITIVE correlation: More failures when prices are higher!")
        else:
            print(f"   → NEGATIVE correlation: More failures when prices are lower")

print("\n2. ECONOMIC IMPACT:")
total_lost_revenue = incidents_with_prices['lost_revenue_eur'].sum()
print(f"   Total lost revenue: €{total_lost_revenue:,.0f}")
print(f"   Average lost revenue per incident: €{total_lost_revenue/len(incidents_with_prices):,.0f}")

# Top 5 costliest incidents
print("\n3. TOP 5 MOST EXPENSIVE INCIDENTS:")
print("-" * 80)
print(f"{'Rank':<6} {'Date':<12} {'Region':<12} {'Price (EUR/MWh)':<17} {'Lost Revenue'}")
print("-" * 80)

top_costly = incidents_with_prices.nlargest(5, 'lost_revenue_eur')
for idx, (_, row) in enumerate(top_costly.iterrows(), 1):
    print(f"#{idx:<5} {row['timestamp'].strftime('%Y-%m-%d'):<12} {row['region_name']:<12} "
          f"{row['avg_price']:<17.2f} €{row['lost_revenue_eur']:,.0f}")

print("\n4. PRICE STATISTICS:")
avg_price_all = market_prices_pd['price_eur_mwh'].mean()
avg_price_incident_days = incidents_with_prices['avg_price'].mean()
print(f"   Average price (all days): €{avg_price_all:.2f}/MWh")
print(f"   Average price (incident days): €{avg_price_incident_days:.2f}/MWh")
if avg_price_incident_days > avg_price_all:
    premium = ((avg_price_incident_days / avg_price_all) - 1) * 100
    print(f"   → Incidents occur when prices are {premium:.0f}% HIGHER than average!")
    print(f"   → Poor timing amplifies financial impact")
else:
    print(f"   → Incidents do not preferentially occur during high-price periods")

print("\n5. KEY INSIGHT:")
if total_lost_revenue > 50000:
    print(f"   Significant economic impact: Over €50k in lost revenue opportunity")
if avg_price_incident_days > avg_price_all * 1.1:
    print(f"   Double penalty: Failures occurring during expensive electricity periods")
    print(f"   Recommendation: Enhanced monitoring during price spikes")
else:
    print(f"   Failures appear random relative to price - not demand-driven")

print("="*80)

In [0]:
df_operations.explain(True)